# Milestone 1: The Foundation

## Goal
Build the basic ingestion layer for the Northstar Support Copilot. We must take raw customer emails and extract structured `intent`, `urgency`, and `sentiment` schemas using Pydantic contracts and zero-shot prompts.

In [ ]:
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

class CustomerEmail(BaseModel):
    intent: str = Field(description="What the customer wants (e.g., Refund, Tech Support, Question)")
    urgency: str = Field(description="HIGH, MEDIUM, or LOW")
    sentiment: str = Field(description="POSITIVE, NEUTRAL, or NEGATIVE")


In [ ]:
def extract_email_data(email_text: str) -> CustomerEmail:
    prompt = f"Extract the intent, urgency, and sentiment from this customer email.\n\nEmail:\n{email_text}"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=CustomerEmail,
        )
    )
    return CustomerEmail.model_validate_json(response.text)

email = "My server just crashed and I'm losing thousands of dollars! Help me NOW!"
result = extract_email_data(email)
print(result)
